# Lesson 21 Lab — Safe Pruning for Detection and Segmentation

**Puzzle:** Can an unchanged average metric hide a large regression on small objects or a rare mask class?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Detection and segmentation heads consume multi-scale features, and business risk is rarely uniform across sizes and classes. A pruning candidate can preserve an aggregate proxy while degrading the feature-pyramid level responsible for small objects. Safety therefore requires slice metrics and per-branch budgets.


## 0. Predict before running

1. Predict which pyramid branch is most sensitive for the small-object proxy.
2. Construct an example where mean error falls but worst-slice error rises.
3. Choose both aggregate and slice-level rollback gates.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A three-scale feature-pyramid toy head, synthetic large/medium/small targets, a uniform-pruning candidate, a protected-high-resolution candidate, and per-slice reconstruction errors form the controlled lab.

- Multi-scale branches have different semantic responsibilities.
- Aggregate quality can pass while a protected slice fails.
- Risk-weighted budgets require explicit slice thresholds.


## 2. Derive the mechanism

High-resolution pyramid features carry more spatial positions and often serve small objects. If aggregate loss weights every tensor element or sample uniformly, a large branch can dominate or a rare slice can disappear in the mean. Define `E_slice` separately and an acceptance rule such as `max slice regression <= tau` in addition to aggregate change. Budget allocation then becomes risk-weighted rather than purely parameter-weighted.

### Mechanism at a glance

```mermaid
flowchart LR
  I["input image"] --> B["pruned backbone"]
  B --> P["multi-scale neck / FPN"]
  P --> H1["classification + box heads"]
  P --> H2["mask / segmentation head"]
  H1 --> O["post-processing"]
  H2 --> O
  O --> E["slice quality + end-to-end latency"]
```

### Walk it step by step

1. **Map the whole task graph.** Detection and segmentation couple backbone features to neck scales, heads, anchors, masks, and post-processing dimensions.
2. **Protect task-sensitive interfaces.** Keep feature pyramid channel agreements, spatial resolutions, class outputs, and mask geometry valid.
3. **Evaluate task slices.** Measure small, medium, and large objects or class and boundary slices—not only an aggregate score.
4. **Include pre- and post-processing.** The deployment gate uses end-to-end latency because NMS, resizing, and mask decoding may dominate after pruning.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 21
LESSON_TITLE = 'Safe Pruning for Detection and Segmentation'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260829
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | uniform pruning across three feature-pyramid branches |
| Candidate | risk-weighted pruning that protects the high-resolution/small-object branch |
| Held constant | feature tensors, targets, total retained-channel budget, head weights, seed, and slice definitions |
| Measurements | aggregate error, large/medium/small slice error, worst-slice regression, and retained channels |
| Evidence | `numerical-model` |

**Experiment:** Compare uniform channel pruning with a high-resolution-protected budget at equal total retained channels.


## 5. Read the experiment code

The notebook constructs target outputs so each slice depends most strongly on its corresponding scale. Both candidates spend the same total channel budget, but allocate it differently. Reporting every slice next to the aggregate exposes whether the protected policy trades average error for a safer worst case.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
channels=24; samples=512
features={"large":torch.randn(samples,channels,device=DEVICE)*0.8,"medium":torch.randn(samples,channels,device=DEVICE),"small":torch.randn(samples,channels,device=DEVICE)*1.8}
weights={name:torch.randn(channels,8,device=DEVICE) for name in features}
targets={name:features[name]@weights[name] for name in features}
def select(counts):
    outputs={}; retained=0
    for name,count in counts.items():
        score=(features[name].square().mean(0).sqrt()[:,None]*weights[name].abs()).sum(1)
        keep=torch.topk(score,count).indices; outputs[name]=features[name][:,keep]@weights[name][keep]; retained+=count
    return outputs,retained
uniform,nu=select({"large":12,"medium":12,"small":12})
protected,np=select({"large":6,"medium":10,"small":20})
def errors(outputs):
    per={name:tensor_metrics(targets[name],outputs[name])["rmse"] for name in targets}
    aggregate=float(torch.sqrt(torch.mean(torch.cat([(outputs[n]-targets[n]).flatten() for n in targets]).square())).item())
    return per,aggregate
ue,ua=errors(uniform); pe,pa=errors(protected)
metrics={"uniform_aggregate_rmse":ua,"protected_aggregate_rmse":pa,"uniform_large_rmse":ue["large"],"uniform_medium_rmse":ue["medium"],"uniform_small_rmse":ue["small"],"protected_large_rmse":pe["large"],"protected_medium_rmse":pe["medium"],"protected_small_rmse":pe["small"],"uniform_worst_slice":max(ue,key=ue.get),"protected_worst_slice":max(pe,key=pe.get),"retained_channels":nu,"protected_allocation":{"large":6,"medium":10,"small":20}}
analysis=(f"Both policies retained {nu} channels across three branches. Uniform allocation produced aggregate RMSE {ua:.6f} "
          f"and small-slice RMSE {ue['small']:.6f}; protecting the high-resolution branch produced {pa:.6f} and "
          f"{pe['small']:.6f}, respectively. The per-slice table—not the aggregate alone—determines whether the risk trade is acceptable.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Uniform aggregate RMSE | 3.688635 |
| Protected aggregate RMSE | 2.925958 |
| Uniform small-slice RMSE | 5.187524 |
| Protected small-slice RMSE | 2.448860 |
| Worst uniform slice | small |
| Total retained channels | 36 |


## 7. Interpret rather than merely print

Both policies retained 36 channels across three branches. Uniform allocation produced aggregate RMSE 3.688635 and small-slice RMSE 5.187524; protecting the high-resolution branch produced 2.925958 and 2.448860, respectively. The per-slice table—not the aggregate alone—determines whether the risk trade is acceptable.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 21,
    "title": 'Safe Pruning for Detection and Segmentation',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Safe pruning treats the worst critical slice as a first-class constraint rather than trusting an aggregate metric.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 21,
  "title": "Safe Pruning for Detection and Segmentation",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260829
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "uniform_aggregate_rmse": 3.6886346340179443,
    "protected_aggregate_rmse": 2.9259581565856934,
    "uniform_large_rmse": 2.372267484664917,
    "uniform_medium_rmse": 2.8775017261505127,
    "uniform_small_rmse": 5.187524318695068,
    "protected_large_rmse": 3.126690626144409,
    "protected_medium_rmse": 3.1481080055236816,
    "protected_small_rmse": 2.4488604068756104,
    "uniform_worst_slice": "small",
    "protected_worst_slice": "medium",
    "retained_channels": 36,
    "protected_allocation": {
      "large": 6,
      "medium": 10,
      "small": 20
    }
  },
  "analysis": "Both policies retained 36 channels across three branches. Uniform allocation p

## 9. Make the bounded decision

> Safe pruning treats the worst critical slice as a first-class constraint rather than trusting an aggregate metric.

**Acceptance/rollback:** Accept only when aggregate detection/segmentation quality and every business-critical size/class slice stay within frozen thresholds.

**Failure analysis:** A synthetic reconstruction proxy is not COCO AP, mask AP, recall, or calibration. Slice definitions chosen after observing failures can overfit the report. Feature channels also interact across the neck and head in real architectures.


## 10. Extend the evidence

Run the policy on a real detector with COCO `AP`, `AP_S`, `AP_M`, `AP_L`, class recall, and mask metrics, then bind each gate to a rollback action.

The full evidence boundary and references are in [`README.md`](README.md).
